<a href="https://colab.research.google.com/github/paridhietal/TransferLearning/blob/main/T%3DtransferLearning4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [ ]:
import json
import chromadb # Import chromadb

# Initialize ChromaDB client and get the collection
# You might need to adjust this based on your ChromaDB setup (e.g., persistent client)
client = chromadb.Client() # Or chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="my_collection") # Replace "my_collection" with your actual collection name

all_memory = collection.get(include=["metadatas"])
training_data = []

for meta in all_memory["metadatas"]:
  if float(meta["score"]) >= 7:
    training_data.append({
        "prompt": meta["task"],
        "completion": f"{meta['output']}\n\nLesson: {meta['lesson']} "
    })

#Save as JSONL
with open("training_data.jsonl", "w") as f:
  for entry in training_data:
    f.write(json.dumps(entry) + "\n")

print(f"Training examples: {len(training_data)}")

Training examples: 0


In [ ]:
!pip install transformers peft datasets accelerate bitsandbytes

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

#Load base model (small, Colab-friendly)
model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: MistralForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'

In [ ]:
#LoRA = only trains a small adapter on top of frozen base model
#Much faster and cheaper than full fine-tuning

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# -> trainable params: -4M out of 7B (0.06%) - very efficient

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollaboratorForLanguageModeling
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

training_args = TrainingArguments(
    output_dir="./fine_tuned_memory_modedl",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)

)

trainer.train()

In [ ]:
#Save adapter weights (small file, not full model)
model.save_pretrained("./memory_adapter")
tokenizer.save_pretrained("./memory_adapter")

#Download to keep safe
import shutil
shutil.make_archive("memory_adapter_backup", "zip", "./memory_adapter")
from google.colab import files
files.download("memory_adapter_backup.zip")

In [ ]:
#Using it for inference
def generate_with_finetuned_model(query):
  inputs = tokenizer(query, return_tensors="pt")
  outputs = model.generate(
      **inputs,
      max_new_tokens=300,
      temperature=0.7,
      do_sample=True
)
  return tokenizer.decode(outputs[0], skip_special_tokens=True)

def full pipeline(query):
  #Get memory context(RAG)
  prompt, _ = build_prompt_with_memory(query)
  #Run through fine-tuned model
  return generate_with_finetuned_model(prompt)